# Loan Prediction - Data Science with Python

This notebook follows along with the "Data Science with Python" tutorial. The goal is to look at a bank's loan application data and build a simple model that predicts whether a loan will be approved (`Loan_Status`) based on things like income, credit history, and marital status.

Every section below has comments explaining what the code does and why, in plain English.

## 1. Import the libraries we need

Before doing anything, we need to bring in the Python libraries that give us tools for handling data (pandas, numpy), making charts (matplotlib, seaborn), and building/testing a machine learning model (scikit-learn).

In [ ]:
# pandas lets us load and work with our data as a table (called a DataFrame)
import pandas as pd

# numpy gives us extra math tools, useful for working with numbers and arrays
import numpy as np

# matplotlib and seaborn are used to make charts/graphs so we can visualize the data
import matplotlib.pyplot as plt
import seaborn as sns

# these are scikit-learn tools we'll use later to encode data, scale it,
# split it into training/testing sets, build a model, and check how good it is
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

# this just makes our charts show up nicely inside the notebook
%matplotlib inline


## 2. Load the dataset

We're using a CSV file (`loan_data.csv`) that contains past loan applications and whether each one was approved or not. `pd.read_csv()` reads that file into a DataFrame, which is basically a spreadsheet we can work with in code.

In [ ]:
# read the CSV file into a DataFrame called df (short for "data frame")
df = pd.read_csv("loan_data.csv")

# show the first 5 rows so we can see what the data actually looks like
df.head()


## 3. Get a first look at the data

Before cleaning or modeling anything, it's good practice to understand what you're working with: how many rows/columns there are, what type of data is in each column, and basic statistics for the numeric columns.

In [ ]:
# .shape tells us (number of rows, number of columns)
print("Rows and columns:", df.shape)

# .info() shows column names, how many non-empty values each has, and their data type
df.info()


In [ ]:
# .describe() gives quick statistics (mean, min, max, etc.) for the numeric columns
# this helps us spot anything unusual, like weirdly high incomes or negative values
df.describe()


## 4. Check for missing values

Real-world data is almost never perfectly clean. Some applicants might not have filled out every field. We need to know exactly where the gaps are before we decide how to handle them.

In [ ]:
# .isnull() marks every empty cell as True, and .sum() adds those up per column
# this tells us exactly how many missing values each column has
df.isnull().sum()


## 5. Explore the data (EDA)

EDA stands for "Exploratory Data Analysis." Basically, we're just looking around the data with some simple counts and charts to understand patterns before we build a model. For example: how many loans were approved vs. rejected? Does being married or having good credit history seem to matter?

In [ ]:
# value_counts() counts how many times each unique value shows up in a column
# here we check how many loans were Approved (Y) vs Not Approved (N)
df['Loan_Status'].value_counts()


In [ ]:
# a simple bar chart showing the approved vs not-approved counts
sns.countplot(x='Loan_Status', data=df)
plt.title("Loan Approval Counts")
plt.show()


In [ ]:
# let's see if credit history has any relationship with loan approval
# crosstab builds a little table comparing two columns against each other
pd.crosstab(df['Credit_History'], df['Loan_Status'])


## 6. Handle the missing values

Machine learning models can't work with empty/missing cells, so we need to fill them in with a reasonable value.

- For text/category columns (like Gender or Self_Employed), we fill gaps with the **mode** (the most common value in that column) — it's the safest guess.
- For number columns (like LoanAmount), we fill gaps with the **median** (the middle value) since it isn't thrown off by extreme outliers the way an average can be.

In [ ]:
# fill missing category columns with the most common value (the "mode") in that column
df['Gender'] = df['Gender'].fillna(df['Gender'].mode()[0])
df['Marital_Status'] = df['Marital_Status'].fillna(df['Marital_Status'].mode()[0])
df['Dependents'] = df['Dependents'].fillna(df['Dependents'].mode()[0])
df['Self_Employed'] = df['Self_Employed'].fillna(df['Self_Employed'].mode()[0])
df['Loan_Amount_Term'] = df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].mode()[0])
df['Credit_History'] = df['Credit_History'].fillna(df['Credit_History'].mode()[0])

# fill the missing LoanAmount values with the median loan amount
df['LoanAmount'] = df['LoanAmount'].fillna(df['LoanAmount'].median())

# double check: this should now show 0 missing values everywhere
df.isnull().sum()


## 7. Clean up the "Dependents" column

The Dependents column has a value of `3+` (meaning "3 or more"), which is text, not a number. Since we want all our columns to be numeric for the model, we'll convert `3+` into just the number `3`.

In [ ]:
# replace the text "3+" with the number 3, then convert the whole column to integers
df['Dependents'] = df['Dependents'].replace('3+', 3).astype(int)

# quick check that it worked
df['Dependents'].unique()


## 8. Convert categorical (text) columns into numbers

Machine learning models only understand numbers, not words like "Male" or "Urban". `LabelEncoder` converts each text category into a corresponding number behind the scenes (e.g. Male -> 1, Female -> 0).

In [ ]:
# create one LabelEncoder we can reuse for each text column
le = LabelEncoder()

# list of all the columns that contain text categories instead of numbers
categorical_columns = ['Gender', 'Marital_Status', 'Graduate', 'Self_Employed', 'Property_Area', 'Loan_Status']

# loop through each one and turn its text values into numbers
for col in categorical_columns:
    df[col] = le.fit_transform(df[col])

# let's peek at the data now that everything is numeric
df.head()


## 9. Split the data into features (X) and target (y)

`X` holds all the columns we'll use to make a prediction (income, credit history, etc.). `y` is the thing we're actually trying to predict — whether the loan was approved.

We also drop `Loan_ID` since it's just a unique identifier and has no real predictive value.

In [ ]:
# X = everything except the ID column and the answer column (what we're predicting)
X = df.drop(['Loan_ID', 'Loan_Status'], axis=1)

# y = the column we're trying to predict: was the loan approved or not
y = df['Loan_Status']


## 10. Scale the numeric columns

Columns like `ApplicantIncome` have much bigger numbers than columns like `Credit_History` (which is just 0 or 1). If we don't scale them, the model can end up giving too much importance to the columns with bigger numbers just because they're bigger. `StandardScaler` puts all these numeric columns on a similar scale.

In [ ]:
scaler = StandardScaler()

# these are the columns with large, varying numeric ranges that benefit from scaling
numeric_columns = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount']

X[numeric_columns] = scaler.fit_transform(X[numeric_columns])

X.head()


## 11. Split into training and testing sets

We don't want to test our model on the same data it learned from — that wouldn't tell us anything useful. So we split the data: 80% to train the model, 20% to test it afterward on data it hasn't seen yet.

In [ ]:
# random_state=42 just makes sure we get the same split every time we run this,
# so our results are repeatable
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


## 12. Build and train the model

We're using **Logistic Regression**, a common and simple algorithm for yes/no (approved/not approved) style predictions. `.fit()` is where the model actually "learns" the patterns from our training data.

In [ ]:
# create the logistic regression model
model = LogisticRegression(max_iter=1000)

# train it on our training data
model.fit(X_train, y_train)


## 13. Make predictions and check accuracy

Now we ask the trained model to predict loan approval for the test set (data it's never seen before), then compare its predictions to the real answers to see how well it did.

In [ ]:
# use the trained model to predict outcomes for the test set
predictions = model.predict(X_test)

# accuracy_score tells us what percentage of predictions were correct
accuracy = accuracy_score(y_test, predictions)
print(f"Model Accuracy: {accuracy * 100:.2f}%")


In [ ]:
# a confusion matrix shows us exactly where the model was right and wrong:
# rows = actual answers, columns = predicted answers
cm = confusion_matrix(y_test, predictions)
print("Confusion Matrix:")
print(cm)

# visualize it as a heatmap so it's easier to read
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


## 14. Wrap-up

Our logistic regression model predicts loan approval with a reasonable level of accuracy using basic applicant info like income, marital status, and credit history. Credit history in particular tends to be one of the strongest signals for approval, which lines up with common sense — lenders care a lot about someone's track record of paying back debt.